# AmpleHate on ViHSD: PhoBERT Baseline

This notebook implements the original **AmpleHate** method (Lee et al., EMNLP 2025) on the **ViHSD** Vietnamese hate-speech dataset using **PhoBERT** as the encoder.

AmpleHate amplifies target-context relationships by:
1. Extracting explicit targets via NER
2. Computing HeadAttention between the [CLS] token and each target token
3. Injecting the summed attention signal back into [CLS] before classification

**Important:** This is a direct port of the original English AmpleHate code to ViHSD/PhoBERT. No Vietnamese-specific improvements are made at this stage — see the final section "Baseline Limitations and Next Steps" for what to improve later.

In [ ]:
!pip install transformers datasets sentencepiece huggingface_hub underthesea easydict -q

In [ ]:
import os, re, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() > 2 else 0

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Workers: {NUM_WORKERS} | Pin memory: {PIN_MEMORY}')

## 2. Hyperparameters

Key differences from original AmpleHate:
- `MODEL_NAME`: PhoBERT instead of BERT-base-uncased
- `MAX_LEN`: 128 (vs 512 in original) for Kaggle T4 efficiency
- `e`: 1.0 (middle of original search range [0.5, 1.5])
- `WARMUP_RATIO`: added for stability (not in original, but standard practice)
- `NUM_CLASSES`: 2 (NON-HATE=0, HATE=1) — same as original binary setting

In [ ]:
# Encoder
MODEL_NAME   = 'vinai/phobert-base'
NER_MODEL    = 'dbmdz/bert-large-cased-finetuned-conll03-english'  # English NER (original AmpleHate)
MAX_LEN      = 128
HIDDEN_DIM   = 768
HEAD_DIM     = HIDDEN_DIM   # head_attention projection dim = encoder hidden dim

# AmpleHate injection strength (e in original paper)
E_INJECTION  = 1.0   # original paper tunes [0.5, 0.75, 1.0, 1.25, 1.5]

# Training
BATCH_SIZE   = 16    # original AmpleHate default
NUM_EPOCHS   = 6     # original AmpleHate uses 4; +2 for ViHSD convergence
LR           = 2e-5  # original AmpleHate default
HEAD_LR      = 5e-5  # higher LR for HeadAttention + classifier head
WARMUP_RATIO = 0.06
DROPOUT      = 0.1   # applied to final_embedding before classifier
PATIENCE     = 2
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.05

# Labels
NUM_CLASSES  = 2
LABEL_NAMES  = ['NON-HATE', 'HATE']

# Checkpointing
CKPT_NAME    = 'best_amplehate_phobert_vihsd.pt'
PLOT_TITLE   = 'AmpleHate (PhoBERT) — ViHSD Baseline'

## 3. Load ViHSD Dataset

Loading from HuggingFace Hub. Requires a Kaggle secret `HF_TOKEN`.
Original ViHSD has 3 labels: CLEAN=0, OFFENSIVE=1, HATE=2.
We remap to binary: NON-HATE=0 (CLEAN+OFFENSIVE), HATE=1 (HATE only).
This matches the mapping used in the PhoBERT-CNN baseline.

In [ ]:
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import huggingface_hub

secret_value = UserSecretsClient().get_secret("HF_TOKEN")
huggingface_hub.login(token=secret_value, add_to_git_credential=False)

ds       = load_dataset("sonlam1102/vihsd")
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Columns: {train_df.columns.tolist()}")
print(f"Label distribution (raw): {train_df['label_id'].value_counts().to_dict()}")


## 4. Label Mapping

ViHSD original: CLEAN=0, OFFENSIVE=1, HATE=2  
Binary mapping: NON-HATE=0 (CLEAN+OFFENSIVE merged), HATE=1 (HATE only)

In [ ]:
train_df['label_id'] = train_df['label_id'].map(lambda x: 1 if x == 2 else 0)
val_df['label_id']   = val_df['label_id'].map(lambda x: 1 if x == 2 else 0)
test_df['label_id']  = test_df['label_id'].map(lambda x: 1 if x == 2 else 0)

label_map = {0: 'NON-HATE', 1: 'HATE'}
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_id'].value_counts().sort_index().rename(label_map)
    print(f"  {name}: {dist.to_dict()}")


## 5. Vietnamese Text Preprocessing

Standard Vietnamese NLP preprocessing: teencode normalization, URL/email/phone removal,
repeated character collapsing, underthesea word tokenization.
Reused from the PhoBERT-CNN baseline notebook unchanged.

In [ ]:
from underthesea import word_tokenize

TEENCODE_MAP = {
    'ko': 'không', 'kh': 'không', 'khong': 'không', 'kg': 'không',
    'hok': 'không', 'hk': 'không', 'hem': 'không', 'kô': 'không',
    'chx': 'chưa', 'chua': 'chưa',
    'r': 'rồi', 'rui': 'rồi', 'ròi': 'rồi', 'oy': 'rồi', 'uj': 'rồi',
    'mk': 'mình', 'mik': 'mình', 'mh': 'mình',
    'tui': 'tôi', 'tau': 'tao',
    'may': 'mày', 'mi': 'mày',
    'bn': 'bạn', 'ban': 'bạn',
    'no': 'nó', 'mng': 'mọi người', 'mn': 'mọi người', 'ae': 'anh em',
    'dc': 'được', 'đc': 'được', 'dk': 'được', 'đk': 'được',
    'đươc': 'được', 'duoc': 'được',
    'vs': 'với', 'voi': 'với',
    'j': 'gì', 'zì': 'gì', 'zi': 'gì',
    'ntn': 'như thế nào', 'nso': 'như sao',
    'biet': 'biết', 'bit': 'biết', 'hieu': 'hiểu', 'nghi': 'nghĩ',
    'muon': 'muốn', 'hoac': 'hoặc', 'neu': 'nếu', 'nen': 'nên',
    'giet': 'giết', 'chui': 'chửi', 'danh': 'đánh',
    'nx': 'nhưng', 'nhg': 'nhưng', 'nhưg': 'nhưng', 'nma': 'nhưng mà',
    'cx': 'cũng', 'cg': 'cũng', 'cung': 'cũng', 'cũg': 'cũng',
    'ms': 'mới', 'boi': 'bởi',
    'oke': 'ok', 'okie': 'ok', 'okê': 'ok', 'okey': 'ok',
    'uh': 'ừ', 'uk': 'ừ', 'uhm': 'ừ',
    'yep': 'đúng', 'yup': 'đúng',
    'haha': 'haha', 'hehe': 'hehe', 'hihi': 'hehe', 'huhu': 'buồn',
    'haiz': 'thở dài', 'haizz': 'thở dài',
    'wtf': 'cái gì vậy', 'omg': 'ôi trời',
    'lol': 'buồn cười', 'lmao': 'buồn cười',
    'fck': 'chửi thề', 'fk': 'chửi thề', 'gg': 'xong rồi', 'ez': 'dễ',
    'bt': 'bình thường', 'bth': 'bình thường',
    'noob': 'tệ', 'nub': 'tệ',
    'xàm': 'vô nghĩa', 'nhảm': 'vô nghĩa',
    'pro': 'giỏi',
    'vl': 'vãi lồn', 'vcl': 'vãi cái lồn', 'vkl': 'vãi kép lồn',
    'vll': 'vãi lồn', 'vleu': 'vãi lồn', 'vloz': 'vãi lồn',
    'dm': 'đụ má', 'đm': 'đụ má', 'd.m': 'đụ má', 'đ.m': 'đụ má',
    'đmm': 'đụ má mày', 'dmm': 'đụ má mày',
    'đtm': 'địt mẹ', 'dtm': 'địt mẹ',
    'cl': 'cái lồn', 'lon': 'lồn', 'loz': 'lồn', 'l0n': 'lồn',
    'đéo': 'không', 'deo': 'không', 'éo': 'không',
    'cc': 'cái con', 'thg': 'thằng',
    'ngu': 'ngu', 'đần': 'đần độn', 'khùng': 'điên', 'dien': 'điên',
    'cút': 'cút', 'cut': 'cút',
    'câm': 'câm miệng', 'im mồm': 'câm miệng',
    'fb': 'facebook', 'yt': 'youtube', 'tt': 'tiktok', 'zl': 'zalo',
    'ig': 'instagram', 'cmt': 'bình luận', 'rep': 'trả lời',
    'vn': 'việt nam', 'hn': 'hà nội', 'hcm': 'hồ chí minh', 'sg': 'sài gòn',
    'iu': 'yêu', 'ieu': 'yêu',
}

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b0[0-9]{9,10}\b', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [TEENCODE_MAP.get(w, w) for w in text.split()]
    return ' '.join(words)

def preprocess(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return text

sample = 'mày ko hiểu gì hết, ngu vcl!!'
print(f'Original : {sample}')
print(f'Processed: {preprocess(sample)}')


## 6. Apply Preprocessing

In [ ]:
print("Preprocessing texts...")
for df, name in [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]:
    df['text_processed'] = df['free_text'].apply(preprocess)
    print(f'  {name}: done')

print("\nSample:")
for _, row in train_df.sample(3, random_state=42).iterrows():
    lbl = label_map[row['label_id']]
    print(f'  [{lbl:>8}] {row["free_text"][:50]}')
    print(f'           -> {row["text_processed"][:50]}')


## 7. PhoBERT Tokenizer

PhoBERT uses a RoBERTa-style BPE tokenizer. Word-segmented Vietnamese text
(from underthesea) maps well to PhoBERT's vocabulary.

In [ ]:
print('Loading PhoBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size:,}')

sample = 'mày ko hiểu gì hết, ngu vcl!!'
print(f'\nOriginal : {sample}')
print(f'Processed: {preprocess(sample)}')
print(f'Tokens   : {tokenizer.tokenize(preprocess(sample))}')

## 8. AmpleHate Target Identification: English NER

**Original AmpleHate uses `dbmdz/bert-large-cased-finetuned-conll03-english`.**

This is an English NER model that extracts entities of types:
ORG, NORP, GPE, LOC, EVENT.

**Limitation for ViHSD:** ViHSD contains Vietnamese text. The English NER
will detect very few entities. Most samples will fall back to using the
[CLS] token (index 0) as the implicit target — the same fallback used in
the original AmpleHate code when no explicit targets are found.

This means the baseline behaves close to a standard PhoBERT classifier
for most Vietnamese samples. This is expected for a faithful port of the
original method and is documented in the "Baseline Limitations" section.

We keep this English NER **exactly as-is** from the original codebase.
Vietnamese-specific target extraction strategies are left for future work.

In [ ]:
class NERTagger:
    """English NER tagger from original AmpleHate (dbmdz/bert-large-cased-finetuned-conll03-english)."""
    def __init__(self, model_name=NER_MODEL):
        self.ner_pipeline = pipeline(
            "ner",
            model=model_name,
            aggregation_strategy="simple",
            device=0 if DEVICE.type == 'cuda' else 'cpu'
        )

    def extract_named_entities(self, text):
        entities = self.ner_pipeline(text)
        return [
            e["word"] for e in entities
            if e["entity_group"] in ["ORG", "NORP", "GPE", "LOC", "EVENT"]
        ]


class NERProcessor:
    """Tokenizes text and finds head_token_idx for target entities.
    Falls back to [0] (CLS) when no entities are found (original AmpleHate behavior).
    """
    def __init__(self, tokenizer, ner_tagger=None, use_ner=True):
        self.tokenizer = tokenizer
        self.ner_tagger = ner_tagger
        self.use_ner = use_ner

    def extract_head_tokens(self, text):
        if not self.use_ner:
            return []
        return self.ner_tagger.extract_named_entities(text)

    def tokenize_and_encode(self, text):
        head_tokens = self.extract_head_tokens(text)
        tokens = self.tokenizer.tokenize(text)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN  # 128 for Kaggle efficiency (original uses 512)
        )
        token_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]

        head_token_idx = []
        for ht in head_tokens:
            try:
                idx = tokens.index(ht) + 1  # +1 for [CLS] at position 0
                if idx < MAX_LEN - 1:        # guard: must be within truncated sequence
                    head_token_idx.append(idx)
            except ValueError:
                continue

        if not head_token_idx:
            head_token_idx = [0]  # CLS fallback (original AmpleHate behavior)

        return token_ids, head_token_idx, attention_mask

### Load English NER Model

In [ ]:
print("Loading English NER model (original AmpleHate)...")
print(f"Model: {NER_MODEL}")
ner_tagger = NERTagger()
ner_processor_train = NERProcessor(tokenizer, ner_tagger=ner_tagger, use_ner=True)
ner_processor_eval  = NERProcessor(tokenizer, ner_tagger=None, use_ner=False)

# Verify on a sample (English text likely to trigger NER)
test_texts = [
    "Tell China to stop bullying Taiwan",     # likely: GPE
    "I hate those people from Vietnam",       # GPE
    "mày ngu vcl",                            # Vietnamese — likely: no NER hit
    "thằng đó là người Hà Nội",              # Vietnamese — likely: no NER hit
]
print("\nNER entity extraction check:")
for t in test_texts:
    entities = ner_tagger.extract_named_entities(t)
    print(f"  {t!r:55s} -> {entities if entities else '[CLS fallback]'}")

## 9. AmpleHate Dataset and DataLoader

The AmpleHate dataset extends the standard text dataset with `head_token_idx`:
a list of token positions for NER-detected entities (or [0] as CLS fallback).

During training: NER is applied to find explicit targets.
During validation/test: NER is disabled (use_ner=False) → always CLS fallback.
This matches the original AmpleHate implementation exactly.

`collate_fn` pads `head_token_idx` to the maximum number of entities in the batch
(zero-padding = CLS position, which is safe for the model).

In [ ]:
class AmpleHateDataset(Dataset):
    """AmpleHate dataset: adds head_token_idx from NER target extraction."""
    def __init__(self, df, ner_processor):
        self.texts  = df['text_processed'].fillna('').tolist()
        self.labels = df['label_id'].astype(int).tolist()
        self.processor = ner_processor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        token_ids, head_token_idx, attention_mask = self.processor.tokenize_and_encode(self.texts[idx])
        return {
            'input_ids':      torch.tensor(token_ids,      dtype=torch.long),
            'head_token_idx': torch.tensor(head_token_idx, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate_fn(batch):
    """Pads head_token_idx to max entity count in the batch."""
    max_heads = max(len(item['head_token_idx']) for item in batch)
    padded_heads = []
    for item in batch:
        h = item['head_token_idx']
        pad = torch.zeros(max_heads - len(h), dtype=torch.long)
        padded_heads.append(torch.cat([h, pad]))
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'head_token_idx': torch.stack(padded_heads),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
    }

In [ ]:
print("Building datasets...")
print("  Train: applying English NER (slow — ~30-60 min on T4)...")
train_ds = AmpleHateDataset(train_df, ner_processor_train)
print("  Val/Test: NER disabled (CLS fallback only)...")
val_ds   = AmpleHateDataset(val_df,   ner_processor_eval)
test_ds  = AmpleHateDataset(test_df,  ner_processor_eval)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, generator=g,
    num_workers=0, pin_memory=PIN_MEMORY  # NER pipeline is not fork-safe
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
def ner_coverage_statistics(dataset, name="dataset"):
    total = len(dataset)
    ner_applied = 0
    print(f"Computing NER coverage for {name} ({total} samples)...")
    for i in range(total):
        item = dataset[i]
        # NER applied if head_token_idx is NOT just [0] (i.e., not pure CLS fallback)
        if not (len(item['head_token_idx']) == 1 and item['head_token_idx'][0].item() == 0):
            ner_applied += 1
    print(f"  NER entities found in: {ner_applied}/{total} ({ner_applied/total*100:.2f}%)")
    print(f"  CLS fallback used for: {total-ner_applied}/{total} ({(total-ner_applied)/total*100:.2f}%)")

# NOTE: This call runs NER on all ~11k training samples (~45 min on T4).
# Skip this cell or move it to after training to avoid doubling NER overhead.
ner_coverage_statistics(train_ds, "Train")

## 10. AmpleHate Model: HeadAttention + PhoBERT

Direct port of the original AmpleHate model (`model/model.py`) with PhoBERT
replacing BERT-base-uncased. Architecture unchanged:

1. **HeadAttention**: Q,V from [CLS]; K from target entity token.
   `score = softmax(W_q·CLS · (W_k·target)^T / sqrt(d))`
   `output = score · W_v·CLS`

2. **Forward pass**:
   ```
   cls = PhoBERT([CLS] token embedding)
   for each entity token at head_token_idx:
       head_attn += HeadAttention(cls, entity_token)
   final = cls + e * head_attn
   logits = Linear(Dropout(final))
   ```
   
When all head_token_idx = 0 (CLS fallback), the model still differs from
plain PhoBERT because HeadAttention applies learned W_q, W_k, W_v projections.

In [ ]:
class HeadAttention(nn.Module):
    """Original AmpleHate HeadAttention (model/model.py:5-27, unchanged)."""
    def __init__(self, hidden_dim, head_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_dim   = head_dim
        self.softmax    = nn.Softmax(dim=-1)
        self.W_q = nn.Linear(hidden_dim, head_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, head_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, head_dim, bias=False)

    def forward(self, cls_embedding, head_token_embedding):
        Q_h = self.W_q(cls_embedding)          # [batch, head_dim]
        K_h = self.W_k(head_token_embedding)   # [batch, head_dim]
        V_h = self.W_v(cls_embedding)          # [batch, head_dim]
        scores  = torch.matmul(Q_h, K_h.T) / (self.head_dim ** 0.5)
        # NOTE: scores is [B,B] -- each sample attends to all others in the batch.
        # At batch_size=1 softmax returns [[1.0]], bypassing attention (original behavior).
        scores  = scores.float()
        weights = self.softmax(scores)
        return torch.matmul(weights, V_h)      # [batch, head_dim]


class AmpleHatePhoBERT(nn.Module):
    """AmpleHate with PhoBERT encoder (original CustomBERT with BERT→PhoBERT swap)."""
    def __init__(self, model_name, hidden_dim=HIDDEN_DIM, e=E_INJECTION, dropout=DROPOUT):
        super().__init__()
        self.bert           = AutoModel.from_pretrained(model_name)
        self.hidden_dim     = hidden_dim
        self.e              = e
        self.head_attention = HeadAttention(hidden_dim, HEAD_DIM)
        self.dropout        = nn.Dropout(dropout)
        self.classifier     = nn.Linear(hidden_dim, NUM_CLASSES)

    def forward(self, input_ids, head_token_idx, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Gather embeddings at each head_token position — [batch, max_heads, hidden]
        expanded_idx = head_token_idx.unsqueeze(-1).expand(-1, -1, self.hidden_dim)
        head_token_embeddings = torch.gather(outputs.last_hidden_state, 1, expanded_idx)

        # Sum HeadAttention over all entity positions (original AmpleHate loop)
        outputs_list = [
            self.head_attention(cls_embedding, head_token_embeddings[:, i, :])
            for i in range(head_token_embeddings.shape[1])
        ]
        head_attention_output = sum(outputs_list)

        # Direct injection: cls + e * attention_output (AmpleHate core contribution)
        final_embedding = cls_embedding + head_attention_output * self.e
        final_embedding = self.dropout(final_embedding)

        return self.classifier(final_embedding)

In [ ]:
class ContrastiveLossCosine(nn.Module):
    """Original AmpleHate contrastive loss (model/cl_loss.py, unchanged)."""
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        # NOTE: requires batch_size >= 2; denominator is batch*(batch-1)
        batch_size  = embeddings.size(0)
        cosine_sim  = F.cosine_similarity(
            embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=-1
        )  # [batch, batch]
        labels      = labels.unsqueeze(1)
        label_matrix = (labels != labels.T).float()
        pos_loss    = (1 - label_matrix) * (1 - cosine_sim)
        neg_loss    = label_matrix * F.relu(cosine_sim - self.margin)
        return (pos_loss + neg_loss).sum() / (batch_size * (batch_size - 1))

In [ ]:
print('Loading AmpleHate + PhoBERT model...')
model = AmpleHatePhoBERT(MODEL_NAME, hidden_dim=HIDDEN_DIM, e=E_INJECTION, dropout=DROPOUT).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Encoder         : {MODEL_NAME}')
print(f'e (injection)   : {E_INJECTION}')
print(f'Dropout         : {DROPOUT}')

## 11. Loss, Optimizer, and Scheduler

Following the original AmpleHate training setup:
- **Loss**: CrossEntropy (primary). The original also supports contrastive loss
  but we use CE-only for this baseline.
- **Optimizer**: AdamW, lr=2e-5 (original default).
- **Class weights**: added to handle ViHSD class imbalance (~89% NON-HATE).
- **Scheduler**: linear warmup added (not in original, but standard for PhoBERT).
- **Differential LR**: higher LR for HeadAttention + classifier head (not in original).

In [ ]:
label_counts  = train_df['label_id'].value_counts().sort_index().values
class_weights = torch.tensor(
    len(train_df) / (NUM_CLASSES * label_counts),
    dtype=torch.float32, device=DEVICE
)
print('Class weights:', class_weights.cpu().numpy().round(3))

criterion_train = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
criterion_eval  = nn.CrossEntropyLoss()

no_decay    = ['bias', 'LayerNorm.weight']
bert_decay, bert_no_decay = [], []
for name, param in model.bert.named_parameters():
    if not param.requires_grad:
        continue
    (bert_no_decay if any(nd in name for nd in no_decay) else bert_decay).append(param)

head_params = (
    list(model.head_attention.parameters()) +
    list(model.classifier.parameters())
)

optimizer = optim.AdamW([
    {'params': bert_decay,    'lr': LR,      'weight_decay': WEIGHT_DECAY},
    {'params': bert_no_decay, 'lr': LR,      'weight_decay': 0.0},
    {'params': head_params,   'lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY},
])

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}')

## 12. Training and Evaluation Loops

Training loop follows the original AmpleHate `train_epoch` and `evaluate` functions.
Key differences from original:
- AMP (mixed precision) added for Kaggle T4 efficiency.
- Gradient clipping added (norm=1.0) for stability.
- Best threshold search on validation set (from original `best_threshold` function).

In [ ]:
def best_threshold(probs: np.ndarray, labels: np.ndarray,
                   grid=np.linspace(0.05, 0.95, 19)) -> float:
    """Grid-search for best classification threshold on validation set.
    Direct copy of train.py:best_threshold (unchanged).
    """
    best_t, best_f1 = 0.5, 0.0
    for t in grid:
        f1 = f1_score(labels, (probs >= t).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss = total_correct = total_n = 0

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)
        y     = batch['labels'].to(DEVICE,          non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        preds = logits.argmax(1)
        total_loss    += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total_n       += y.size(0)

    return total_loss / total_n, total_correct / total_n

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = total_n = 0
    all_probs, all_labels = [], []

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)
        y     = batch['labels'].to(DEVICE,          non_blocking=PIN_MEMORY)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)
            loss   = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)[:, 1]

        total_loss    += loss.item() * y.size(0)
        total_n       += y.size(0)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    t      = best_threshold(all_probs, all_labels)
    y_pred = (all_probs >= t).astype(int)
    acc    = accuracy_score(all_labels, y_pred)
    macro_f1 = f1_score(all_labels, y_pred, average='macro', zero_division=0)

    return total_loss / total_n, acc, macro_f1, float(t)

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     [], 'threshold': []
}
best_f1, best_epoch, best_t_saved, patience_counter = -1.0, 0, 0.5, 0

hdr = f"{'Epoch':>6} | {'Tr Loss':>8} | {'Tr Acc':>7} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val F1':>6} | {'Thresh':>6} | {'LR':>8} | {'Time':>6}"
print(hdr)
print('-' * len(hdr))

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion_train, scaler
    )
    vl_loss, vl_acc, vl_f1, vl_t = evaluate(model, val_loader, criterion_eval)
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_f1'].append(vl_f1)
    history['threshold'].append(vl_t)

    flag = ''
    if vl_f1 > best_f1:
        best_f1, best_epoch, best_t_saved = vl_f1, epoch, vl_t
        torch.save({'model': model.state_dict(), 'threshold': vl_t}, CKPT_NAME)
        patience_counter = 0
        flag = ' *saved*'
    else:
        patience_counter += 1

    print(
        f'{epoch:>6} | {tr_loss:>8.4f} | {tr_acc:>7.4f} | '
        f'{vl_loss:>8.4f} | {vl_acc:>7.4f} | {vl_f1:>6.4f} | '
        f'{vl_t:>6.2f} | {current_lr:>8.2e} | {elapsed:>5.1f}s{flag}'
    )

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest Val Macro-F1 = {best_f1:.4f} at epoch {best_epoch} (threshold={best_t_saved:.2f})')

## 13. Training Curves

In [ ]:
n            = len(history['train_loss'])
epochs_range = range(1, n + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_range, history['val_f1'], 'g-o')
axes[2].axvline(best_epoch, color='red', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[2].set_title('Val Macro F1'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.suptitle(PLOT_TITLE, fontsize=14)
plt.tight_layout()
plt.savefig('training_curves_amplehate.png', dpi=150)
plt.show()

## 14. Test Set Evaluation

Loading best checkpoint and evaluating on the held-out test set.
Threshold from validation grid search is applied.

In [ ]:
ckpt = torch.load(CKPT_NAME, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
threshold = ckpt['threshold']
print(f'Loaded checkpoint from epoch {best_epoch}, threshold={threshold:.2f}')

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, threshold):
    model.eval()
    all_probs, all_labels = [], []

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)
        probs = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

    y_pred = (np.array(all_probs) >= threshold).astype(int)
    return np.array(all_labels), y_pred

y_true, y_pred = get_predictions(model, test_loader, threshold)
print('Classification Report — Test Set')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

## 15. Full Test Metrics

In [ ]:
acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro',    zero_division=0)
macro_p  = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_r  = recall_score(y_true, y_pred, average='macro',    zero_division=0)
hate_f1  = f1_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)

print(f"Accuracy           : {acc:.4f}")
print(f"Macro Precision    : {macro_p:.4f}")
print(f"Macro Recall       : {macro_r:.4f}")
print(f"Macro F1           : {macro_f1:.4f}")
print(f"F1 (HATE class)    : {hate_f1:.4f}")

In [ ]:
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm,     annot=True, fmt='d',   cmap='Blues',   ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
axes[0].set_title('Confusion Matrix (count)')
axes[1].set_title('Confusion Matrix (% per row)')
for ax in axes:
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.suptitle(PLOT_TITLE + ' — Test Set', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_amplehate.png', dpi=150)
plt.show()

In [ ]:
os.makedirs('outputs', exist_ok=True)

config = {
    'notebook'         : 'vihsd-amplehate-phobert-baseline',
    'method'           : 'AmpleHate (original, English NER)',
    'encoder'          : MODEL_NAME,
    'ner_model'        : NER_MODEL,
    'max_len'          : MAX_LEN,
    'hidden_dim'       : HIDDEN_DIM,
    'e_injection'      : E_INJECTION,
    'dropout'          : DROPOUT,
    'num_classes'      : NUM_CLASSES,
    'label_names'      : LABEL_NAMES,
    'lr_encoder'       : float(LR),
    'lr_head'          : float(HEAD_LR),
    'best_epoch'       : int(best_epoch),
    'best_val_f1'      : round(float(best_f1), 4),
    'best_threshold'   : round(float(threshold), 2),
    'test_accuracy'    : round(float(acc), 4),
    'test_macro_f1'    : round(float(macro_f1), 4),
    'test_macro_p'     : round(float(macro_p), 4),
    'test_macro_r'     : round(float(macro_r), 4),
    'test_f1_hate'     : round(float(hate_f1), 4),
}

with open('outputs/amplehate_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(json.dumps(config, indent=2, ensure_ascii=False))

## 16. Baseline Limitations and Next Steps

This notebook faithfully ports the original AmpleHate to ViHSD/PhoBERT.
The following known limitations explain why performance may be suboptimal
for Vietnamese and point to improvements for future work.

---

### Limitation 1: English NER on Vietnamese Text

**What the original does:** Uses `dbmdz/bert-large-cased-finetuned-conll03-english`
to extract named entities (ORG, NORP, GPE, LOC, EVENT) as explicit hate targets.

**Problem:** ViHSD contains Vietnamese text. The English NER model was trained
on English CoNLL-2003 data and has near-zero recall on Vietnamese named entities.
As a result, the vast majority of training samples fall back to the CLS fallback
(head_token_idx = [0]), meaning the HeadAttention acts on the [CLS] token itself.

**NER coverage estimate:** Typically < 5% of ViHSD samples will trigger real NER hits.

**Next step:** Replace the English NER with a Vietnamese NER model (e.g.,
`NlpHUST/ner-vietnamese-electra-base` or `vinai/PhoNER_COVID19`) and adapt
entity types to Vietnamese hate speech targets (ethnicity groups, political entities,
gender terms, etc.).

---

### Limitation 2: Target Entity Types Not Aligned with ViHSD

**What the original does:** Targets ORG, NORP, GPE, LOC, EVENT — entity types
commonly targeted in English implicit hate speech (races as NORP, countries as GPE).

**Problem:** ViHSD hate speech frequently targets gender, religion, occupation,
and regional groups expressed as common nouns (e.g., "thằng", "bọn", "tụi nó")
which are not named entities and will never be captured by any NER model.

**Next step:** Add a Vietnamese hate target lexicon (target group keywords),
dependency-based target extraction, or a dedicated Vietnamese target tagger.

---

### Limitation 3: Word Segmentation Mismatch

**What the original does:** Tokenizes raw English text directly with BERT tokenizer.
The NER model also processes raw text, so entity spans map to the same tokenized form.

**Problem:** We apply underthesea word segmentation before PhoBERT tokenization
(required for PhoBERT), but the English NER model runs on raw (non-segmented) text.
After word segmentation, underscores appear in compound words (e.g., "việt_nam"),
so NER entity positions may not align with PhoBERT token positions. The `tokens.index(ht)`
lookup in `NERProcessor.tokenize_and_encode` may fail silently more often.

**Next step:** When switching to a Vietnamese NER, run it on segmented text
(or use a word-segmentation-aware model like PhoBERT-based NER) so entity spans
align with PhoBERT tokenization.

---

### Limitation 4: max_length Reduced from 512 to 128

**What the original does:** max_length=512 for full sentence coverage.

**Trade-off:** Reduced to 128 for Kaggle T4 VRAM efficiency. Most ViHSD comments
are short (median < 50 PhoBERT tokens) so truncation rarely occurs in practice.

**Next step:** Profile truncation rate; increase max_length if needed.

---

### Summary: What a Strong Vietnamese AmpleHate Would Need

| Component | Current (Baseline) | Next Step |
|---|---|---|
| NER model | English (dbmdz/bert-large-cased-finetuned-conll03-english) | Vietnamese NER |
| Target types | ORG, NORP, GPE, LOC, EVENT | Hate-relevant groups + lexicons |
| NER coverage | ~1-5% of ViHSD | Should cover >30% |
| Encoder | PhoBERT (correct) | Keep or try PhoBERT-large |
| Tokenization | underthesea + PhoBERT | Keep |

Despite these limitations, this baseline establishes a reproducible reference point
for measuring the impact of future Vietnamese-specific improvements.